# Understanding customer value and retention

**The first order gives us useful clues, but it cannot reliably tell us who will become a repeat customer.**

## Follow the evidence

1. Clean the data and keep an audit trail
2. Explore price, quantity and typical purchases
3. Which countries, products and customers buy the most?
4. Compare customer value and repeat purchases by country
5. Who keeps customers the longest?
6. When should we expect a second purchase?
7. What is a customer worth after their first purchase?
8. Does the first order tell us who will return?
9. Do country and season change the picture?
10. How much can we predict after one order and after two?
11. What should the business do?

The reproducible analysis cell is collapsed by default. It rebuilds the tables, charts and aggregate files from both workbook sheets. The chapters below use the same results and sequence as the public project page. No customer-level records are exported.

In [1]:
"""Rebuild a data-first, single-column retail analysis and executable notebook."""
from pathlib import Path
import html
import json
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import nbformat
from nbconvert import HTMLExporter
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, roc_auc_score

OUT = Path.cwd()
CONCLUSION='The first order gives us useful clues, but it cannot reliably tell us who will become a repeat customer.'

def first_basket_membership(lines, first_orders):
    """Flag each physical product once per customer's first invoice, without changing sales lines."""
    keys=first_orders.reset_index()[['customer','invoice']]
    matched=lines.merge(keys,on=['customer','invoice'],how='inner',validate='many_to_one')
    return matched.loc[matched.stock.astype(str).str.match(r'^\d'),['customer','stock']].drop_duplicates().reset_index(drop=True)

def product_return_summary(membership, outcomes, minimum=50):
    """Aggregate overlapping first-product groups and Wilson intervals; publish no customer IDs."""
    joined=membership.drop_duplicates(['customer','stock']).join(outcomes[['repeat90']],on='customer').dropna(subset=['repeat90'])
    summary=joined.groupby('stock').agg(Customers=('customer','nunique'),returned=('repeat90','sum'))
    summary=summary.loc[summary.Customers.ge(minimum)].copy()
    n=summary.Customers.to_numpy(dtype=float);p=summary.returned.to_numpy(dtype=float)/n
    z=1.96;denominator=1+z*z/n
    center=(p+z*z/(2*n))/denominator
    radius=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/denominator
    summary['Returned within 90 days (%)']=100*p
    summary['Lower 95% (%)']=100*np.maximum(0,center-radius)
    summary['Upper 95% (%)']=100*np.minimum(1,center+radius)
    return summary.rename(columns={'returned':'Returned customers'}).sort_values('Customers',ascending=False)

def second_order_snapshot(orders,end):
    """Compare first and second baskets at the second decision time; later orders are labels only."""
    ordered=orders.sort_values(['customer','date','invoice'])
    first=ordered.groupby('customer',sort=False).head(1).set_index('customer')
    later=ordered.join(first.date.rename('first_date'),on='customer')
    second=later[later.date.gt(later.first_date)].groupby('customer',sort=False).head(1).set_index('customer')
    snapshot=first[['date','revenue','units','products']].rename(columns={'date':'first_date','revenue':'first_value','units':'first_units','products':'first_variety'}).join(second[['date','revenue','units','products']].rename(columns={'date':'second_date','revenue':'second_value','units':'second_units','products':'second_variety'}),how='inner')
    snapshot=snapshot[snapshot.second_date.le(pd.Timestamp(end)-pd.Timedelta(days=90))].copy()
    snapshot['gap_days']=(snapshot.second_date-snapshot.first_date).dt.total_seconds()/86400
    followup=ordered.join(snapshot.second_date,on='customer')
    followup=followup[followup.date.gt(followup.second_date)&followup.date.lt(followup.second_date+pd.Timedelta(days=90))]
    snapshot['next_purchase_90d']=followup.groupby('customer').size().reindex(snapshot.index,fill_value=0).gt(0).astype(int)
    return snapshot

def run():
    OUT.mkdir(exist_ok=True)
    raw = pd.concat(pd.read_excel(OUT/'data/online_retail_II.xlsx', sheet_name=None).values(), ignore_index=True)
    steps = []
    def record(label, frame):
        steps.append({'Cleaning step':label, 'Rows remaining':len(frame)})
    record('Raw workbook: both annual sheets', raw)
    identical_lines=int(raw.duplicated().sum())
    df=raw.copy(); record('Identical lines noted and retained; no duplicate removal',df)
    df=df.rename(columns={'Invoice':'invoice','StockCode':'stock','Description':'description','Quantity':'quantity','InvoiceDate':'date','Price':'price','Customer ID':'customer','Country':'country'})
    df['date']=pd.to_datetime(df['date'],errors='coerce')
    df=df[df.date.notna() & df.price.gt(0) & df.quantity.ne(0)].copy();record('Keep valid dates, positive prices and nonzero quantities',df)
    df['invoice']=df.invoice.astype(str);df['stock']=df.stock.astype(str)
    df['return']=df.invoice.str.upper().str.startswith('C') | df.quantity.lt(0)
    df['revenue']=np.where(df['return'],-abs(df.quantity*df.price),df.quantity*df.price)
    identified=df[df.customer.notna()].copy(); record('Customer analysis: require a customer identifier',identified)
    sales=df[~df['return']].copy()
    buys=identified[~identified['return']].copy()
    orders=buys.groupby(['customer','invoice']).agg(date=('date','min'),country=('country','first'),revenue=('revenue','sum'),units=('quantity','sum'),products=('stock','nunique')).reset_index()
    orders=orders.sort_values(['customer','date','invoice'])
    customer=orders.groupby('customer').agg(first=('date','min'),last=('date','max'),orders=('invoice','size'),gross=('revenue','sum'),country=('country','first'))
    customer['net']=identified.groupby('customer').revenue.sum()
    customer['span_days']=(customer['last']-customer['first']).dt.total_seconds()/86400
    customer['repeat']=customer.orders.ge(2)
    customer['average_order']=customer.gross/customer.orders
    end=df.date.max()
    eligible=customer[customer['first']<=end-pd.Timedelta(days=180)].copy()
    joined=orders.join(customer['first'],on='customer')
    joined['age']=(joined.date-joined['first']).dt.total_seconds()/86400
    eligible['repeat90']=joined[joined.age.between(0,90,inclusive='left')].groupby('customer').invoice.nunique().reindex(eligible.index,fill_value=0).ge(2)
    eligible['active180']=joined[joined.age.between(150,180,inclusive='left')].groupby('customer').size().reindex(eligible.index,fill_value=0).gt(0)
    netlines=identified.join(customer['first'],on='customer')
    age=(netlines.date-netlines['first']).dt.total_seconds()/86400
    eligible['revenue180']=netlines[age.between(0,180,inclusive='left')].groupby('customer').revenue.sum().reindex(eligible.index,fill_value=0)
    countries=customer.groupby('country').agg(customers=('orders','size'),repeat_customers=('repeat','sum'),repeat_share=('repeat','mean'),net_revenue=('net','sum'),average_orders=('orders','mean'),average_observed_value=('net','mean'),median_observed_value=('net','median'),median_observed_span_days=('span_days','median'))
    countryorders=orders.groupby('country').agg(orders=('invoice','size'),gross=('revenue','sum'))
    countries['average_order_value']=countryorders.gross/countryorders.orders
    standardized=eligible.groupby('country').agg(eligible_customers=('orders','size'),repeat90=('repeat90','mean'),active180=('active180','mean'),average180=('revenue180','mean'))
    countries=countries.join(standardized)
    stable=countries[countries.eligible_customers.ge(50)].sort_values('average180',ascending=False)
    gaps=orders.groupby('customer').date.diff().dt.total_seconds()/86400
    positivegaps=gaps[gaps.gt(0)]
    daily=orders.assign(day=orders.date.dt.normalize()).drop_duplicates(['customer','day']).sort_values(['customer','day'])
    daygaps=daily.groupby('customer').day.diff().dt.days.dropna()
    product_sales=sales[sales.stock.str.match(r'^\d')]
    products=product_sales.groupby('stock').agg(description=('description','first'),units=('quantity','sum'),revenue=('revenue','sum'),invoice_count=('invoice','nunique')).sort_values('units',ascending=False)
    descriptions=[]
    for label,series in [('Unit price (£), all valid sales lines',sales.price),('Units per sales line',sales.quantity),('Order value (£), identified customers',orders.revenue),('Units per order',orders.units),('Orders per customer',customer.orders),('Observed net revenue per customer (£)',customer.net)]:
        d=series.describe(percentiles=[.25,.5,.75,.95,.99]);descriptions.append({'Measure':label,**d.to_dict()})
    statistics=pd.DataFrame(descriptions).rename(columns={'count':'Count','mean':'Mean','std':'Std dev','min':'Minimum','25%':'25th percentile','50%':'Median','75%':'75th percentile','95%':'95th percentile','99%':'99th percentile','max':'Maximum'})
    sns.set_theme(style='whitegrid')
    def bucket_chart(series,edges,names,file,title,xlabel,ylabel,right=False):
        counts=pd.cut(series,edges,labels=names,right=right,include_lowest=True).value_counts(sort=False)
        assert counts.sum()==len(series),f'Uncounted values in {file}'
        fig,ax=plt.subplots(figsize=(15,6))
        counts.plot.bar(ax=ax,color='#2456a6',width=.75)
        ax.set(title=title,xlabel=xlabel,ylabel=ylabel,ylim=(0,None))
        ax.tick_params(axis='x',rotation=25);plt.setp(ax.get_xticklabels(),ha='right')
        ax.yaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter('{x:,.0f}'))
        ax.set_title(title,pad=40)
        ax.bar_label(ax.containers[0],labels=[f'{int(x):,}' for x in counts],padding=3,fontsize=8)
        ax.margins(y=.12)
        ax.text(0,1.02,'Blue bars count '+ylabel.lower()+'. All values are included.',transform=ax.transAxes,fontsize=11)
        fig.tight_layout();fig.savefig(OUT/file,dpi=150);plt.close(fig)
        return counts
    price_edges=[0,1,2,5,10,20,50,100,500,1000,10000,np.inf]
    price_names=['£0 to under £1','£1 to under £2','£2 to under £5','£5 to under £10','£10 to under £20','£20 to under £50','£50 to under £100','£100 to under £500','£500 to under £1,000','£1,000 to under £10,000','£10,000 or more']
    bucket_chart(sales.price,price_edges,price_names,'prices.png','What does one item cost?','Price of one item (£), grouped into ranges','Sales lines')
    bucket_chart(sales.quantity,[0,1,2,5,10,20,50,100,500,1000,np.inf],['1','2','3–5','6–10','11–20','21–50','51–100','101–500','501–1,000','More than 1,000'],'quantities.png','How many items are bought on each sales line?','Items on one invoice line','Sales lines',right=True)
    bucket_chart(orders.revenue,[0,50,100,250,500,1000,5000,10000,np.inf],['Under £50','£50 to under £100','£100 to under £250','£250 to under £500','£500 to under £1,000','£1,000 to under £5,000','£5,000 to under £10,000','£10,000 or more'],'order-values.png','How much do customers spend per order?','Total value of a completed order (£)','Completed orders')
    bucket_chart(customer.orders,[0,1,2,3,5,10,20,50,100,np.inf],['1 order','2 orders','3 orders','4–5 orders','6–10 orders','11–20 orders','21–50 orders','51–100 orders','More than 100 orders'],'order-volume.png','How often do customers place orders?','Completed orders per customer','Customers',right=True)
    bucket_chart(positivegaps,[0,1,7,14,30,60,90,180,365,np.inf],['Less than 1 day','1 to under 7 days','7 to under 14 days','14 to under 30 days','30 to under 60 days','60 to under 90 days','90 to under 180 days','180 to under 365 days','365 days or more'],'purchase-gaps.png','How long do repeat buyers wait between orders?','Time between consecutive orders; zero-time gaps excluded','Repeat-purchase intervals')
    def bar(series,file,title,xlabel,ylabel,sort=True,percentage=False):
        fig,ax=plt.subplots(figsize=(15,7));values=series.sort_values() if sort else series.iloc[::-1]
        values.plot.barh(ax=ax,color='#2456a6')
        ax.set(title=title,xlabel=xlabel,ylabel=ylabel,xlim=(0,100) if percentage else (0,None))
        ax.set_title(title,pad=40)
        ax.xaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter('{x:,.0f}'))
        if percentage:ax.bar_label(ax.containers[0],labels=[f'{x:.1f}%' for x in values],padding=5,fontsize=11)
        ax.text(0,1.02,'Blue bars show '+xlabel.lower()+'.',transform=ax.transAxes,fontsize=11)
        fig.tight_layout();fig.savefig(OUT/file,dpi=150);plt.close(fig)
    bar(countries.net_revenue.nlargest(12),'country-revenue.png','Which countries generate the most net revenue?','Observed net revenue (£)','Country')
    product_names=products.description.fillna('Unnamed product').astype(str).str.slice(0,40)+' ['+products.index.astype(str)+']'
    product_units=products.units.copy();product_units.index=product_names
    product_revenue=products.revenue.copy();product_revenue.index=product_names
    bar(product_units.nlargest(12),'product-units.png','Which products sell the most items?','Units sold, before returns','Product name [stock code]')
    bar(product_revenue.nlargest(12),'product-revenue.png','Which products generate the highest gross sales?','Gross sales revenue (£), before returns','Product name [stock code]')
    bar(stable.active180.nlargest(12)*100,'country-retention.png','Which countries still purchase near month six?','Customers purchasing on days 150–179 (%)','Country (at least 50 eligible customers)',percentage=True)
    first=orders.groupby('customer',sort=False).head(1).set_index('customer')
    early=eligible.join(first[['revenue','units','products']],rsuffix='_first')
    physical_first_lines=buys[buys.stock.str.match(r'^\d')].merge(first.reset_index()[['customer','invoice']],on=['customer','invoice'],how='inner',validate='many_to_one')
    first_item_counts=physical_first_lines.groupby('customer').agg(units=('quantity','sum'),products=('stock','nunique'))
    early=early.drop(columns=['units','products']).join(first_item_counts)
    early=early[early.products.gt(0)].copy()
    spend_names=['Less than £50','£50 to under £100','£100 to under £250','£250 to under £500','£500 to under £1,000','£1,000 or more']
    early['price_band']=pd.cut(early.revenue,[0,50,100,250,500,1000,np.inf],labels=spend_names,right=False,include_lowest=True)
    bands=early.groupby('price_band',observed=True).agg(customers=('orders','size'),repeat_buyers=('repeat90','sum'),repeat90=('repeat90','mean'),average180=('revenue180','mean'))
    bands.index=bands.index.astype(str)
    bands.index.name='Amount spent on the first order'
    bar(bands.repeat90*100,'first-order.png','After their first order, how many customers buy again?','Customers who placed another order within 90 days (%)','Amount spent on the FIRST order (£)',sort=False,percentage=True)
    readable_bands=bands.rename(columns={'customers':'Customers in this group','repeat_buyers':'Bought again within 90 days','repeat90':'Bought again (%)','average180':'Average first-180-day revenue (£)'})
    readable_bands['Bought again (%)']*=100
    train=early[early['first']<pd.Timestamp('2010-06-01')]
    val=early[(early['first']>=pd.Timestamp('2010-09-01'))&(early['first']<pd.Timestamp('2010-10-01'))]
    test=early[early['first']>=pd.Timestamp('2011-01-01')]
    features=['revenue','units','products'];metrics=[];models={}
    for name,model in [('Logistic regression',make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))),('Gradient boosting',HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,l2_regularization=5,random_state=42)),('Probabilistic baseline',None)]:
        if model is None:p=np.full(len(val),train.repeat90.mean())
        else:model.fit(np.log1p(train[features]),train.repeat90);p=model.predict_proba(np.log1p(val[features]))[:,1]
        metrics.append({'Model':name,'Validation Brier':brier_score_loss(val.repeat90,p),'Validation AUC':roc_auc_score(val.repeat90,p)})
        models[name]=model
    winner=min(metrics,key=lambda m:m['Validation Brier'])['Model'];model=models[winner]
    p=np.full(len(test),train.repeat90.mean()) if model is None else model.predict_proba(np.log1p(test[features]))[:,1]
    final={'winner':winner,'brier':brier_score_loss(test.repeat90,p),'auc':roc_auc_score(test.repeat90,p),'customers':len(test)}
    # Examine what was in the FIRST invoice; later purchases never enter these features.
    membership=first_basket_membership(buys,first)
    membership=membership[membership.customer.isin(early.index)].copy()
    product_summary=product_return_summary(membership,early,minimum=50)
    description_map=products.description.fillna('Unnamed product').astype(str).to_dict()
    product_summary.insert(0,'Product',product_summary.index.map(lambda code:description_map.get(code,code)))
    product_summary.index.name='Stock code'
    common_codes=membership.stock.value_counts().head(30).index
    pair_rows=[]
    for customer_id,basket in membership[membership.stock.isin(common_codes)].groupby('customer'):
        pair_rows.extend((customer_id,' + '.join(pair)) for pair in combinations(sorted(basket.stock.unique()),2))
    pair_membership=pd.DataFrame(pair_rows,columns=['customer','stock'])
    pair_summary=product_return_summary(pair_membership,early,minimum=50)
    pair_summary.insert(0,'Products bought together',pair_summary.index.map(lambda pair:' + '.join(description_map.get(code,code) for code in pair.split(' + '))))
    pair_summary.index.name='Stock code pair'
    def behavior_groups(values,edges,names):
        temp=early[['repeat90']].copy()
        temp['group']=pd.cut(values,edges,labels=names,include_lowest=True,right=True)
        groups=temp.groupby('group',observed=True).agg(Customers=('repeat90','size'),returned=('repeat90','sum'),rate=('repeat90','mean'))
        assert groups.Customers.sum()==len(early)
        groups['Returned within 90 days (%)']=groups.rate*100
        return groups.drop(columns='rate').rename(columns={'returned':'Returned customers'})
    unit_groups=behavior_groups(early.units,[0,1,5,10,25,50,100,np.inf],['1 item','2–5 items','6–10 items','11–25 items','26–50 items','51–100 items','More than 100 items'])
    variety_groups=behavior_groups(early.products,[0,1,2,5,10,20,50,np.inf],['1 product','2 products','3–5 products','6–10 products','11–20 products','21–50 products','More than 50 products'])
    unit_groups.index.name='Total items in the first order'
    variety_groups.index.name='Different products in the first order'
    bar(unit_groups['Returned within 90 days (%)'],'first-order-units.png','Does buying more items on the first order predict returning?','Customers who placed another order within 90 days (%)','Total items in the FIRST order',sort=False,percentage=True)
    bar(variety_groups['Returned within 90 days (%)'],'first-order-variety.png','Does a wider choice of products predict returning?','Customers who placed another order within 90 days (%)','Different products in the FIRST order',sort=False,percentage=True)
    baseline_rate=float(early.repeat90.mean()*100)
    def product_signal_chart(summary,file,title,pair=False):
        if summary.empty:return False
        shown=summary.head(10).iloc[::-1]
        names=shown['Products bought together' if pair else 'Product'].astype(str)
        if pair:names=names.str.replace(' + ','\n+ ',regex=False)
        names=names+' (n='+shown.Customers.astype(int).astype(str)+')'
        fig,ax=plt.subplots(figsize=(16,10 if pair else 8))
        rate=shown['Returned within 90 days (%)'].to_numpy()
        errors=np.vstack([rate-shown['Lower 95% (%)'],shown['Upper 95% (%)']-rate])
        ax.barh(np.arange(len(shown)),rate,color='#2456a6',xerr=errors,error_kw={'ecolor':'#17233b','capsize':4})
        ax.set_yticks(np.arange(len(shown)),names)
        ax.set(xlabel='Customers who placed another order within 90 days (%)',ylabel='First-order '+('product pair' if pair else 'product')+'; n = customers in group',xlim=(0,100))
        ax.set_title(title,pad=54)
        ax.axvline(baseline_rate,color='#b55420',linestyle='--',linewidth=2)
        ax.text(0,1.02,'Blue: repeat rate · Black: 95% interval · Orange dashed line: all eligible customers ('+f'{baseline_rate:.1f}%'+')',transform=ax.transAxes,fontsize=10)
        fig.tight_layout();fig.savefig(OUT/file,dpi=150);plt.close(fig)
        return True
    product_signal_chart(product_summary,'first-order-products.png','Which common first-order products are associated with returning?')
    pair_chart=product_signal_chart(pair_summary,'first-order-pairs.png','Which common first-order product pairs are associated with returning?',pair=True)
    # Product vocabulary is selected only from customers used to train the models.
    training_counts=membership[membership.customer.isin(train.index)].stock.value_counts()
    vocabulary=training_counts[training_counts.ge(10)].head(40).index.tolist()
    flags=pd.crosstab(membership[membership.stock.isin(vocabulary)].customer,membership[membership.stock.isin(vocabulary)].stock).reindex(index=early.index,columns=vocabulary,fill_value=0).gt(0).astype(float)
    augmented=np.column_stack([np.log1p(early[features].to_numpy()),flags.to_numpy()])
    positions=pd.Series(np.arange(len(early)),index=early.index)
    train_positions=positions.loc[train.index].to_numpy();val_positions=positions.loc[val.index].to_numpy();test_positions=positions.loc[test.index].to_numpy()
    augmented_candidates={
        'Logistic regression + product identities':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),
        'Gradient boosting + product identities':HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,l2_regularization=5,random_state=42),
    }
    augmented_validation=[]
    for name,candidate in augmented_candidates.items():
        candidate.fit(augmented[train_positions],train.repeat90)
        vp=candidate.predict_proba(augmented[val_positions])[:,1]
        augmented_validation.append({'Model':name,'Validation Brier':brier_score_loss(val.repeat90,vp),'Validation AUC':roc_auc_score(val.repeat90,vp)})
    enriched_choice=min(augmented_validation,key=lambda row:row['Validation Brier'])
    enriched_prediction=augmented_candidates[enriched_choice['Model']].predict_proba(augmented[test_positions])[:,1]
    enriched_final={'Model':enriched_choice['Model'],'Test Brier':brier_score_loss(test.repeat90,enriched_prediction),'Test AUC':roc_auc_score(test.repeat90,enriched_prediction)}
    test_comparison=pd.DataFrame([{'Model':winner+' (spend, items, variety)','Test Brier':final['brier'],'Test AUC':final['auc']},enriched_final]).set_index('Model')
    differences=(p-test.repeat90.to_numpy())**2-(enriched_prediction-test.repeat90.to_numpy())**2
    rng=np.random.default_rng(42)
    improvement_interval=np.quantile(differences[rng.integers(0,len(test),size=(2000,len(test)))].mean(axis=1),[.025,.975])
    improvement=float(differences.mean())
    combined_validation=pd.DataFrame(metrics+augmented_validation).set_index('Model')
    selected_extended=min(metrics+augmented_validation,key=lambda row:row['Validation Brier'])['Model']
    product_report={'minimum_customers':50,'eligible_customers':len(early),'baseline_repeat_rate_percent':baseline_rate,'products':product_summary.reset_index().to_dict('records'),'pairs':pair_summary.reset_index().to_dict('records'),'item_count_groups':unit_groups.reset_index().to_dict('records'),'variety_groups':variety_groups.reset_index().to_dict('records'),'product_feature_count':len(vocabulary),'training_product_vocabulary':vocabulary,'selected_on_validation':selected_extended,'validation_models':combined_validation.reset_index().to_dict('records'),'test_comparison':test_comparison.reset_index().to_dict('records'),'brier_improvement':improvement,'brier_improvement_paired_95_ci':improvement_interval.tolist()}
    (OUT/'first-order-product-results.json').write_text(json.dumps(product_report,indent=2),encoding='utf-8')
    # Separate acquisition spending from money earned during the rest of the first 180 days.
    early['later_revenue_180d']=early.revenue180-early.revenue
    later_groups=membership.join(early[['later_revenue_180d']],on='customer').groupby('stock').agg(Customers=('customer','nunique'),average_later=('later_revenue_180d','mean'),median_later=('later_revenue_180d','median'))
    later_groups=later_groups[later_groups.Customers.ge(50)].sort_values('Customers',ascending=False)
    later_groups.insert(0,'Product',later_groups.index.map(description_map))
    later_groups=later_groups.rename(columns={'average_later':'Average later net revenue (£)','median_later':'Median later net revenue (£)'})
    later_groups.index.name='Product code'
    later_values=later_groups.head(10)['Average later net revenue (£)'].copy()
    later_values.index=later_groups.head(10).Product
    bar(later_values,'later-customer-revenue.png','How much revenue follows these common first-order products?','Average later net revenue within 180 days (£)','Product included in the FIRST order')
    # Everyone in this timing comparison has the full follow-up window.
    positive_second=orders.join(customer['first'],on='customer')
    positive_second=positive_second[positive_second.date.gt(positive_second['first'])].groupby('customer',sort=False).head(1).set_index('customer')
    first_to_second=(positive_second.date-customer['first']).dt.total_seconds()/86400
    first_to_second=first_to_second.reindex(early.index)
    returned90=first_to_second[first_to_second.lt(90)]
    checkpoints=[7,14,30,60,90]
    second_timing=pd.DataFrame([{'Days after first order':day,'Customers with a later order':int(first_to_second.lt(day).sum()),'All first-order customers':len(early),'Returned by this point (%)':float(first_to_second.lt(day).mean()*100)} for day in checkpoints])
    fig,ax=plt.subplots(figsize=(15,6))
    ax.plot([0]+checkpoints,[0]+second_timing['Returned by this point (%)'].tolist(),color='#2456a6',marker='o',linewidth=2)
    ax.set(xlabel='Days since first order',ylabel='All eligible customers who have bought again (%)',xlim=(0,90),ylim=(0,100))
    ax.set_title('When does the second purchase happen?',pad=35)
    ax.text(0,1.02,'Blue line: cumulative return percentage; customers without a second order stay in the denominator.',transform=ax.transAxes,fontsize=10)
    fig.tight_layout();fig.savefig(OUT/'time-to-second-order.png',dpi=150);plt.close(fig)
    country_context=early.groupby('country').agg(Customers=('repeat90','size'),Repeat=('repeat90','mean'),first_spend=('revenue','mean'),later_value=('later_revenue_180d','mean'))
    country_context=country_context[country_context.Customers.ge(50)].copy();country_context['Repeat']*=100
    country_context=country_context.rename(columns={'Repeat':'Returned within 90 days (%)','first_spend':'Average first-order spending (£)','later_value':'Average later net revenue (£)'})
    quarter_names={1:'Jan–Mar',2:'Apr–Jun',3:'Jul–Sep',4:'Oct–Dec'}
    early['entry_quarter']=early['first'].map(lambda date:f'{date.year} Q{(date.month-1)//3+1} ({quarter_names[(date.month-1)//3+1]})')
    quarter_context=early.groupby('entry_quarter').agg(Customers=('repeat90','size'),Repeat=('repeat90','mean'),first_spend=('revenue','mean'),later_value=('later_revenue_180d','mean'))
    quarter_context=quarter_context[quarter_context.Customers.ge(50)].copy();quarter_context.Repeat*=100
    quarter_context=quarter_context.rename(columns={'Repeat':'Returned within 90 days (%)','first_spend':'Average first-order spending (£)','later_value':'Average later net revenue (£)'})
    quarter_context.index.name='When the customer first bought'
    bar(quarter_context['Returned within 90 days (%)'],'seasonal-repeat-purchases.png','Do customers starting in different seasons return at the same rate?','Customers who placed another order within 90 days (%)','Quarter of FIRST order; at least 50 customers',sort=False,percentage=True)
    most_common_code=product_summary.index[0]
    common_members=membership[membership.stock.eq(most_common_code)].join(early[['repeat90','entry_quarter']],on='customer')
    common_seasons=common_members.groupby('entry_quarter').agg(Customers=('customer','nunique'),rate=('repeat90','mean'))
    common_seasons=common_seasons[common_seasons.Customers.ge(50)].copy();common_seasons['Returned within 90 days (%)']=100*common_seasons.pop('rate')
    common_seasons.index.name='Quarter of first purchase'
    # Evaluate two information sets on the SAME customers and decision dates.
    physical_counts=buys[buys.stock.str.match(r'^\d')].groupby(['customer','invoice']).agg(units=('quantity','sum'),products=('stock','nunique')).reset_index()
    model_orders=orders.drop(columns=['units','products']).merge(physical_counts,on=['customer','invoice'],how='left').fillna({'units':0,'products':0})
    second_snapshot=second_order_snapshot(model_orders,end)
    second_snapshot=second_snapshot[second_snapshot.index.isin(early.index)&second_snapshot.second_variety.gt(0)].copy()
    second_train=second_snapshot[second_snapshot.second_date.lt(pd.Timestamp('2010-06-01'))]
    second_val=second_snapshot[second_snapshot.second_date.ge(pd.Timestamp('2010-09-01'))&second_snapshot.second_date.lt(pd.Timestamp('2010-10-01'))]
    second_test=second_snapshot[second_snapshot.second_date.ge(pd.Timestamp('2011-01-01'))]
    assert min(len(second_train),len(second_val),len(second_test))>=30
    second_comparison=[];second_predictions={};second_validation=[]
    for information,cols in [('First order only',['first_value','first_units','first_variety']),('First and second orders',['first_value','first_units','first_variety','second_value','second_units','second_variety','gap_days'])]:
        choices={
            'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),
            'Gradient boosting':HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,l2_regularization=5,random_state=42),
            'Training-rate baseline':None,
        }
        validations=[]
        for algorithm,estimator in choices.items():
            if estimator is None:
                vp=np.full(len(second_val),second_train.next_purchase_90d.mean())
            else:
                estimator.fit(np.log1p(second_train[cols]),second_train.next_purchase_90d)
                vp=estimator.predict_proba(np.log1p(second_val[cols]))[:,1]
            validations.append((brier_score_loss(second_val.next_purchase_90d,vp),algorithm))
            second_validation.append({'Information available':information,'Algorithm':algorithm,'Validation Brier':brier_score_loss(second_val.next_purchase_90d,vp)})
        _,algorithm=min(validations)
        sp=np.full(len(second_test),second_train.next_purchase_90d.mean()) if choices[algorithm] is None else choices[algorithm].predict_proba(np.log1p(second_test[cols]))[:,1]
        second_predictions[information]=sp
        second_comparison.append({'Information available':information,'Selected algorithm':algorithm,'Test customers':len(second_test),'Test Brier':brier_score_loss(second_test.next_purchase_90d,sp),'Test AUC':roc_auc_score(second_test.next_purchase_90d,sp)})
    observed=second_test.next_purchase_90d.to_numpy()
    second_difference=(second_predictions['First order only']-observed)**2-(second_predictions['First and second orders']-observed)**2
    second_improvement=float(second_difference.mean())
    second_interval=np.quantile(second_difference[rng.integers(0,len(second_test),size=(2000,len(second_test)))].mean(axis=1),[.025,.975])
    next_steps={'eligible_first_order_customers':len(early),'first_to_second':second_timing.to_dict('records'),'second_order_median_days_among_90d_returners':float(returned90.median()),'later_revenue_mean':float(early.later_revenue_180d.mean()),'later_revenue_median':float(early.later_revenue_180d.median()),'later_revenue_by_first_product':later_groups.reset_index().to_dict('records'),'country_context':country_context.reset_index().to_dict('records'),'season_context':quarter_context.reset_index().to_dict('records'),'common_product':description_map[most_common_code],'common_product_by_season':common_seasons.reset_index().to_dict('records'),'second_order_model_validation':second_validation,'second_order_model_test':second_comparison,'second_order_split_counts':{'train':len(second_train),'validation':len(second_val),'test':len(second_test)},'second_order_brier_improvement':second_improvement,'second_order_improvement_paired_95_ci':second_interval.tolist()}
    (OUT/'retention-next-steps.json').write_text(json.dumps(next_steps,indent=2),encoding='utf-8')
    firstmonth=customer['first'].dt.to_period('M');co=orders.assign(cohort=orders.customer.map(firstmonth),month=orders.date.dt.to_period('M'))
    co['offset']=(co.month.dt.year-co.cohort.dt.year)*12+co.month.dt.month-co.cohort.dt.month
    counts=co.groupby(['cohort','offset']).customer.nunique().unstack(fill_value=0);ret=counts.div(counts[0],axis=0)
    lastmonth=end.to_period('M')
    for cohort in ret.index:
        for offset in ret.columns:
            if cohort+int(offset)>=lastmonth:ret.loc[cohort,offset]=np.nan
    fig,ax=plt.subplots(figsize=(11,8));sns.heatmap(ret.iloc[:,:13],ax=ax,cmap='Blues',vmin=0,vmax=1,cbar_kws={'label':'Share of original customers purchasing that month (0–1)'})
    ax.set(xlabel='Months since first observed purchase',ylabel='First observed purchase month',title='Customer retention by starting month');fig.tight_layout();fig.savefig(OUT/'cohort-retention.png',dpi=150);plt.close(fig)
    money=lambda x:f'£{x:,.2f}'
    labels={'customers':'Customers','repeat_customers':'Repeat customers','repeat_share':'Repeat share (0–1)','net_revenue':'Net revenue (£)','average_orders':'Orders per customer','average_observed_value':'Observed value/customer (£)','median_observed_value':'Median value/customer (£)','median_observed_span_days':'Median first-to-last span (days)','average_order_value':'Average order value (£)','eligible_customers':'Complete 180-day customers','repeat90':'Repeat within 90 days (0–1)','active180':'Active days 150–179 (0–1)','average180':'180-day value/customer (£)','description':'Description','units':'Units sold','revenue':'Gross sales (£)','invoice_count':'Invoices'}
    labels['country']='Country'
    table=lambda frame:'<div class="table-scroll" tabindex="0">'+frame.reset_index().rename(columns=labels).to_html(index=False,border=0,classes='data-table',float_format=lambda x:f'{x:,.2f}',formatters={column:lambda value:f'{value:.4f}' for column in frame.columns if 'Brier' in str(column) or 'AUC' in str(column)})+'</div>'
    sections=[]
    def section(title,body):sections.append((title,body))
    def image(file,caption):return f'<figure><img src="{file}" alt="{html.escape(caption)}"><figcaption>{caption}</figcaption></figure>'
    section('1. Clean the data and keep an audit trail',f'<p>A sales line records one product on an invoice. We combine a customer’s positive sales lines on the same invoice into one completed order. Returns reduce revenue and never count as new orders.</p>{table(pd.DataFrame(steps).set_index("Cleaning step"))}<div class="finding"><strong>{identical_lines:,} lines match earlier lines exactly. All are retained.</strong><p>Matching fields do not tell us whether a line is a recording error or a legitimate repeated item. We flag the matches rather than remove them.</p></div><p>Sales without customer identifiers remain in product and price summaries. Customer comparisons require an identifier. Nonpositive prices, zero quantities and invalid dates are excluded. Extreme positive prices and quantities remain in the analysis.</p>')
    section('2. Explore price, quantity and typical purchases',f'<p>All prices are in pounds sterling. Unit price means one item’s price; order value means the whole basket. The table shows the full range, including minimums, maximums and unusually large purchases.</p>{table(statistics.set_index("Measure"))}<h3>Item prices across the full range</h3><p>Prices start at £0 and are grouped into named ranges. Each bar counts sales lines in that range. Ranges have different widths, so compare the counts directly; no logarithms are used.</p>'+image('prices.png','Blue counts sales lines in each price range. The £0-to-under-£1 bucket contains positive prices below £1; zero-priced lines are excluded during cleaning. The final bucket includes every price of £10,000 or more.')+image('quantities.png','Blue counts sales lines by the number of items bought. For example, “3–5” means three, four or five items on that line. Larger quantities remain in the final buckets.')+image('order-values.png','Blue counts completed orders by the amount spent on the entire basket. All order values are included, including orders of £10,000 or more.')+image('order-volume.png','Blue counts customers by their number of completed orders across the file. The “1 order” group contains one-time customers.'))
    section('3. Which countries, products and customers buy the most?',image('country-revenue.png','Blue represents net revenue after returns, across identified customers in each country.')+'<h3>Most items sold</h3>'+image('product-units.png','Blue counts items sold. Product names and their stock codes appear together. This ranking measures volume, before returns.')+'<h3>Highest gross sales revenue</h3>'+image('product-revenue.png','Blue shows pounds of gross sales before returns. This answers a different question: which products generate the most money, rather than the most units?')+f'<h3>Customer concentration</h3><p>The top 10% of customers by observed net revenue account for {customer.net.nlargest(max(1,int(len(customer)*.1))).sum()/customer.net.sum():.1%} of identified net revenue. Only aggregate customer groups are published.</p>')
    public_countries=stable.copy()
    public_countries[['repeat_share','repeat90','active180']]*=100
    public_countries=public_countries.rename(columns={'repeat_share':'Repeat customers (%)','repeat90':'Bought again within 90 days (%)','active180':'Buying near month six (%)'})
    section('4. Compare customer value and repeat purchases by country',f'<p>Product costs are unavailable, so we compare net revenue per customer. Countries appear here only if at least <strong>50 customers</strong> have a complete first 180 days in the data.</p><div class="finding"><strong>{html.escape(stable.index[0])} has the highest average first-180-day customer revenue: {money(stable.iloc[0].average180)}.</strong></div><p>“Average order value” is the average basket before returns. “Observed value per customer” is revenue after returns across the file. “180-day value” gives each customer the same amount of follow-up time. Repeat percentages show customers placing a second order.</p>{table(public_countries.sort_values("net_revenue",ascending=False))}')
    section('5. Who keeps customers the longest?',f'<p>The file can show how long customers keep buying during its two years. It cannot tell us when their relationship truly ends. For a fair country comparison, we look at customers with a complete 180 days and ask whether they still buy during days 150–179.</p>'+image('country-retention.png','Blue shows the percentage buying near month six. Only countries with at least 50 customers observed for the full 180 days are included.')+image('cohort-retention.png','Rows group customers by the month they first bought. Columns show months since then. Darker blue means more customers returned; the color bar runs from 0 (none) to 1 (everyone). White means the month is not fully observed.'))
    section('6. How soon do repeat buyers buy again?',f'<p>This chart includes only customers who ordered more than once, and only strictly positive time gaps between their consecutive orders. It excludes {int(gaps.eq(0).sum()):,} same-timestamp gaps.</p><div class="finding"><strong>Median wait between repeat orders: {positivegaps.median():.1f} days.</strong><p>The shortest positive gap is {positivegaps.min()*1440:.0f} minute. Across separate purchase dates, the shortest gap is {daygaps.min():.0f} day. Closely spaced invoices may belong to the same shopping visit.</p></div>'+image('purchase-gaps.png','Blue counts repeat-purchase intervals, not individual customers. A customer with five orders can contribute four intervals. “Less than 1 day” includes positive gaps shorter than 24 hours; zero-time gaps are excluded.'))
    section('7. What is a customer worth?',f'<p>Observed customer value is total sales minus recorded returns. Across identified purchasing customers, average observed value is <strong>{money(customer.net.mean())}</strong> and the median is <strong>{money(customer.net.median())}</strong>. For customers with a complete first 180 days, average value is <strong>{money(eligible.revenue180.mean())}</strong>.</p><p>These are revenue measurements over defined windows. A true lifetime extends beyond the file. The earlier revenue forecast substantially underestimated spending, so its estimates should not determine campaign budgets.</p>')
    example=bands.loc['£250 to under £500']
    section('8. Does the first order tell us who will return?',f'<p>We examine first-order spending, total items, product variety, specific products and pairs. Customers need a complete 180-day follow-up and at least one physical product in their first invoice. Item counts and product flags exclude service codes such as postage; basket spending includes all charges.</p><h3>Does the amount spent matter?</h3><p>Group customers by <strong>what they spent on their first order</strong>. Then count how many placed another order within 90 days. The spending labels describe the first basket, not later spending and not only customers who returned.</p><div class="finding"><strong>How to read one row</strong><p>Among {int(example.customers):,} customers whose first order cost £250 to under £500, {int(example.repeat_buyers):,} bought again within 90 days: <strong>{example.repeat90:.1%}</strong>. The chart’s horizontal bar shows that percentage.</p></div>'+image('first-order.png','Left-hand labels show first-order spending ranges. The horizontal axis shows the percentage who bought again within 90 days. “£1,000 or more” means the FIRST basket cost at least £1,000; later orders can be any size.')+table(readable_bands)+'<p>“Customers in this group” counts everyone in that first-order spending range. “Bought again” counts those with a second order within 90 days. Average 180-day revenue includes the first order, so it naturally increases when the first basket is larger; the repeat percentage is the better test of returning behavior.</p>')
    sections[0]=(sections[0][0],sections[0][1]+'<p><strong>What does “first order” mean?</strong> It means the first order recorded for that customer in this file. Some customers may have purchased before the file begins in December 2009, so the data cannot establish that they were genuinely new customers.</p><h3>Missing values in the raw file</h3>'+table(raw.isna().sum().to_frame('Missing rows')))
    extra='<h3>Does the number of items matter?</h3><p>Total items counts every unit: ten identical mugs count as ten items. The chart asks how many customers returned after buying that quantity on their first order. Counts appear in the table; groups with fewer than 50 customers deserve extra caution.</p>'+image('first-order-units.png','Blue shows the share returning within 90 days, grouped by total units in the first basket. The table gives the number of customers behind each bar.')+table(unit_groups)
    extra+='<h3>Does buying different products matter?</h3><p>Product variety counts different stock codes. Ten identical mugs count as one product; one mug and one plate count as two. Quantity and variety can therefore describe very different shopping behavior.</p>'+image('first-order-variety.png','Blue shows the share returning within 90 days, grouped by the number of different products in the first basket.')+table(variety_groups)
    extra+='<h3>Which specific first-order products are associated with returning?</h3><p>Each row groups customers whose first basket included that product. Their basket may also contain other products. The chart and table show the ten most common qualifying products, requiring at least 50 customers per product. Products are ordered by customer count, not by the most flattering return rate.</p>'+image('first-order-products.png','Blue bars show repeat-purchase percentages. Black whiskers show 95% intervals. The orange dashed line is the repeat rate across all eligible customers. A customer appears in several groups if their first basket contains several products.')+table(product_summary.head(10))+'<details><summary>Explore all '+str(len(product_summary))+' qualifying products</summary>'+table(product_summary)+'</details>'
    extra+='<h3>What about products bought together?</h3><p>A pair means the first basket included both named products; it may contain additional items. We examine pairs among the 30 most common first-order products and require at least 50 customers per pair. Pair groups overlap, so their customer counts must not be added together.</p>'
    if pair_chart:extra+=image('first-order-pairs.png','Blue shows the repeat rate for each common product pair; black shows a 95% interval and orange shows the overall repeat rate. The chart shows the ten most common qualifying pairs.')+table(pair_summary.head(10))+'<details><summary>Explore all '+str(len(pair_summary))+' qualifying product pairs</summary>'+table(pair_summary)+'</details>'
    else:extra+='<p>No product pair met the 50-customer threshold. There is insufficient evidence to rank combinations reliably.</p>'
    extra+='<p>These are associations. Country, season, basket size and wholesale buying can influence both product choice and returning. Buying a particular product is not proven to cause a second purchase. The 95% intervals describe sampling variation for each group; they do not correct for trying many products.</p>'
    prediction_body='<h3>What can we predict after the first order?</h3><p>Logistic regression and gradient boosting first use basket spending, total items and product variety. A simple baseline assigns every customer the training repeat rate. Two additional models also receive a yes/no flag for each of up to 40 common products, selected using training customers only. Rare and unfamiliar products receive no flag.</p><p>Customers entering before June 2010 train the models; September 2010 selects the winner; January 2011 onward tests it. The gaps allow earlier 90-day outcomes to become available before the next stage. Brier measures probability error (lower is better). AUC measures ranking, from 0.5 for chance to 1 for perfect ranking.</p>'+table(combined_validation)+'<p>The next table compares the validation-selected model without product identities against the validation-selected model with them, on the same later customers.</p>'+table(test_comparison)
    direction='improved' if improvement>0 else 'worsened'
    uncertainty='The interval includes zero, so the apparent difference is uncertain.' if improvement_interval[0]<=0<=improvement_interval[1] else 'The interval stays on one side of zero for these fixed models.'
    prediction_body+=f'<div class="finding"><strong>Adding product identities {direction} the later-customer Brier score by {abs(improvement):.4f}.</strong><p>Brier error reduction: {improvement:.4f}; paired 95% interval {improvement_interval[0]:.4f} to {improvement_interval[1]:.4f}. Positive means improvement. {uncertainty}</p></div><p>Validation selects <strong>'+html.escape(selected_extended)+'</strong> from the five candidates. The later test period has already been used for earlier versions of this project; a new period would be needed for a completely fresh confirmation. A product signal should guide further testing rather than automatically determine retention spending.</p>'
    sections[7]=(sections[7][0],sections[7][1]+extra)
    concentration=[]
    for fraction in [.01,.05,.10,.25]:
        count=max(1,int(len(customer)*fraction));group=customer.nlargest(count,'net')
        concentration.append({'Customer group':f'Top {fraction:.0%} by net revenue','Customers':count,'Share of net revenue':group.net.sum()/customer.net.sum(),'Average orders':group.orders.mean(),'Average observed value (£)':group.net.mean()})
    sections[2]=(sections[2][0],'<p>Product rankings include stock codes starting with a digit. Letter-only service codes, such as manual charges and postage, are excluded from these two rankings but remain in overall revenue totals.</p>'+sections[2][1]+table(pd.DataFrame(concentration).set_index('Customer group')))
    timing_body=f'<p>A second order is the first invoice recorded strictly after the first order’s timestamp. Same-timestamp invoices are excluded from this timing view. Every customer here has a complete 90-day follow-up; customers who never return still count in the total.</p><div class="finding"><strong>{second_timing.iloc[-1]["Returned by this point (%)"]:.1f}% placed a later-timestamp order within 90 days.</strong><p>Among those who returned within that window, the median wait was {returned90.median():.1f} days. That median describes returners only, not everyone who made a first purchase.</p></div>'+image('time-to-second-order.png','Blue shows the cumulative percentage of all eligible first-order customers who have placed a later order by each checkpoint. The rows below are cumulative and must not be added together.')+table(second_timing.set_index('Days after first order'))+'<p>The natural timing of purchases suggests windows for testing follow-up messages. It does not establish that a reminder at those dates would increase sales. This stricter timestamp measure can differ slightly from the invoice-based repeat rate used in the first-order comparisons.</p><h3>How do the gaps between all repeat orders compare?</h3>'
    sections[5]=('6. When should we expect a second purchase?',timing_body+sections[5][1])
    sections[6]=('7. What is a customer worth after their first purchase?',f'<p>Across identified customers, average observed net revenue is {money(customer.net.mean())}; the median is {money(customer.net.median())}. These amounts cover each customer’s observed history, not a complete lifetime.</p><h3>Separate the first sale from later customer value</h3><p>For each eligible customer, we take net revenue during their first 180 days and subtract the original first basket’s sales. This prevents an expensive first order from automatically making the customer look valuable afterwards. Returns recorded during the window remain deducted, including refunds of the first order, so later net revenue can be negative.</p><div class="finding"><strong>Average later net revenue: {money(early.later_revenue_180d.mean())} per customer.</strong><p>The median is {money(early.later_revenue_180d.median())}. Compare both: a small number of large buyers can raise the average substantially.</p></div>'+image('later-customer-revenue.png','Blue shows average net revenue during the rest of the first 180 days after subtracting the first basket. The ten most common first-order products are shown; customers can appear in multiple product groups.')+table(later_groups.head(10))+'<details><summary>Explore later revenue for all qualifying first-order products</summary>'+table(later_groups)+'</details><p>The table also shows medians and customer counts. Products are selected by how common they were in first orders, rather than by unusually high revenue results. Product costs are unavailable, so these values measure revenue, not profit.</p>')
    context_body='<p>A product may look promising because it is popular in a particular market or season. These comparisons use complete follow-up windows and require at least 50 customers per row.</p><h3>Country differences</h3>'+table(country_context)+'<h3>Season and year of the first recorded purchase</h3><p>The file starts in December 2009, so 2009 Q4 contains December only and may include previously established customers. The final 2011 Q2 group contains only customers entering early enough to complete the 180-day follow-up. These partial periods and different customer mixes can affect the comparison.</p>'+image('seasonal-repeat-purchases.png','Blue shows 90-day repeat percentages by the year and quarter of the first recorded purchase. Counts and later revenue appear below. Differences can reflect season, existing customers or changing trading conditions; they do not isolate a seasonal effect.')+table(quarter_context)+'<h3>Does the same product show the same pattern throughout the year?</h3><p>For '+html.escape(description_map[most_common_code])+', the next table compares first-order customers entering in different quarters. Rows need at least 50 customers. This is a consistency check for one common product, not a separate model or proof of a seasonal effect.</p>'+table(common_seasons)+'<p>Different return rates across countries and quarters mean that a single overall product percentage should be interpreted cautiously. These comparisons show associations rather than isolating the effect of country, season or product choice.</p>'
    section('9. Do country and season change the picture?',context_body)
    second_direction='improved' if second_improvement>0 else 'worsened' if second_improvement<0 else 'did not change'
    second_caution='The interval includes zero, so this test does not establish a clear improvement.' if second_interval[0]<=0<=second_interval[1] else 'The interval does not include zero for these fixed models and evaluation customers.'
    prediction_body+='<h3>What changes once a customer buys twice?</h3><p>We now ask whether a customer who has already made two purchases will buy again during the next 90 days. Both versions predict the same outcome for the same repeat buyers, starting at the second-order timestamp. One version uses only their original first-basket information. The other also knows the second basket’s value, quantity, variety and the wait between orders. This isolates the value of the additional information; it does not compare repeat buyers against one-time buyers.</p><p>Each version compares logistic regression, gradient boosting and a simple baseline that assigns everyone the training repeat rate. The lowest validation Brier selects the method, so a more complex model must earn its place.</p>'+f'<p>There are {len(second_train):,} training customers, {len(second_val):,} validation customers and {len(second_test):,} later test customers. Splits follow the date of the second order, using the same June/September/January boundaries and 90-day gaps. Each customer has a complete 90-day follow-up after their second order.</p>'+table(pd.DataFrame(second_validation).set_index(['Information available','Algorithm']))+table(pd.DataFrame(second_comparison).set_index('Information available'))+f'<div class="finding"><strong>Second-order information {second_direction} Brier error by {abs(second_improvement):.4f} on this matched group.</strong><p>Error reduction: {second_improvement:.4f}; paired 95% interval {second_interval[0]:.4f} to {second_interval[1]:.4f}. Positive means improvement. {second_caution}</p></div><p>This comparison applies to customers who have bought twice. It cannot tell us which one-time customers would become repeat buyers, and its scores should not be compared directly with the earlier first-order test on a different population.</p>'
    section('10. How much can we predict after one order and after two?',prediction_body)
    second_takeaway='The second-order information improved the matched prediction test, providing a better basis for updating customers who have already returned.' if second_improvement>0 and second_interval[0]>0 else 'The matched second-order test does not establish a clear prediction gain from the additional information.'
    section('11. What should the business do?',f'<div class="finding conclusion"><strong>{CONCLUSION}</strong></div><p>First-order spending, quantity, product variety and product choice show useful patterns, but the later-customer prediction test remains limited. Adding product identities did not establish a clear improvement. Country and seasonal differences also affect how these patterns should be interpreted.</p><p>{second_takeaway} This does not resolve uncertainty about customers who have bought only once.</p><ul><li>Use early signals to form groups for small retention tests, rather than treating them as certain customer outcomes.</li><li>Use the time-to-second-order evidence to choose reminder windows to test against a no-reminder group.</li><li>Judge customer value using later revenue as well as the first sale, and compare medians with averages.</li><li>Update the customer picture after a second purchase, while checking the matched-model evidence above.</li><li>Measure real campaign effects and product costs before making automatic spending decisions.</li></ul><p>The analysis identifies where to investigate and what to test. A new time period and a randomized retention campaign would provide stronger evidence for deployment.</p>')
    document='<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Understanding customer value and retention</title><link rel="stylesheet" href="analysis-story.css"></head><body><nav><a href="../../">Alan Cromlish</a> · <a href="customer-lifetime-value.ipynb" download>Download Jupyter notebook</a> · <a href="analysis.html">Notebook HTML</a></nav><main><header><h1>Understanding customer value and retention</h1><p>Follow the evidence from raw retail transactions to repeat purchasing, customer value and the first signs of a lasting relationship.</p></header>'
    document+='<div class="finding conclusion" id="main-conclusion"><strong>'+CONCLUSION+'</strong><p>Explore how first purchases, return timing, country, season and later revenue build that conclusion.</p></div>'
    document+='<div class="contents"><strong>Explore the analysis</strong>'+''.join(f'<a href="#section-{i}">{html.escape(title)}</a>' for i,(title,_) in enumerate(sections,1))+'</div>'
    document+=''.join(f'<section id="section-{i}"><h2>{title}</h2>{body}</section>' for i,(title,body) in enumerate(sections,1))
    document+='<footer>Source: <a href="https://archive.ics.uci.edu/dataset/502/online%2Bretail">UCI Online Retail II</a>, Chen (2012), CC BY 4.0. Product costs are unavailable. Only aggregate customer results are published.</footer></main></body></html>'
    (OUT/'index.html').write_text(document,encoding='utf-8')
    report={'raw_rows':len(raw),'identical_lines_retained':identical_lines,'cleaning':steps,'customers':len(customer),'minimum_country_customers':50,'country_sample_sizes':stable.eligible_customers.to_dict(),'top_country_180':stable.index[0],'first_order_test':final,'shortest_positive_gap_minutes':positivegaps.min()*1440,'zero_time_gaps_excluded':int(gaps.eq(0).sum()),'first_order_groups':readable_bands.reset_index().to_dict('records')}
    (OUT/'exploration-results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')

    return sections

sections = run()

C:\Users\acrom\AppData\Roaming\Python\Python313\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\acrom\AppData\Roaming\Python\Python313\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python313\Lib\subprocess.py", line 1036, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^

## 1. Clean the data and keep an audit trail

In [2]:
from IPython.display import HTML, display
display(HTML(sections[0][1]))

Cleaning step,Rows remaining
Raw workbook: both annual sheets,1067371
Identical lines noted and retained; no duplicate removal,1067371
"Keep valid dates, positive prices and nonzero quantities",1061164
Customer analysis: require a customer identifier,824293
index,Missing rows
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0


## 2. Explore price, quantity and typical purchases

In [3]:
from IPython.display import HTML, display
display(HTML(sections[1][1]))

Measure,Count,Mean,Std dev,Minimum,25th percentile,Median,75th percentile,95th percentile,99th percentile,Maximum
"Unit price (£), all valid sales lines","1,041,670.00",4.08,51.45,0.00,1.25,2.10,4.13,9.95,18.00,"25,111.09"
Units per sales line,"1,041,670.00",10.96,126.51,1.00,1.00,3.00,10.00,30.00,100.00,"80,995.00"
"Order value (£), identified customers","36,969.00",479.95,"1,374.99",0.38,160.80,305.25,489.26,"1,253.78","3,697.92","168,469.60"
Units per order,"36,969.00",289.60,"1,235.79",1.00,73.00,155.00,292.00,788.60,"2,322.52","87,167.00"
Orders per customer,"5,878.00",6.29,13.01,1.00,1.00,3.00,7.00,21.00,46.00,398.00
Observed net revenue per customer (£),"5,878.00","2,842.69","14,040.15","-10,953.50",335.03,862.75,"2,217.36","9,129.54","26,703.44","598,215.22"


## 3. Which countries, products and customers buy the most?

In [4]:
from IPython.display import HTML, display
display(HTML(sections[2][1]))

Customer group,Customers,Share of net revenue,Average orders,Average observed value (£)
Top 1% by net revenue,58,0.31,84.29,"89,277.14"
Top 5% by net revenue,293,0.51,38.04,"29,157.57"
Top 10% by net revenue,587,0.63,27.23,"18,004.97"
Top 25% by net revenue,1469,0.81,16.62,"9,254.07"


## 4. Compare customer value and repeat purchases by country

In [5]:
from IPython.display import HTML, display
display(HTML(sections[3][1]))

Country,Customers,Repeat customers,Repeat customers (%),Net revenue (£),Orders per customer,Observed value/customer (£),Median value/customer (£),Median first-to-last span (days),Average order value (£),Complete 180-day customers,Bought again within 90 days (%),Buying near month six (%),180-day value/customer (£)
United Kingdom,5349,3875,72.44,"13,865,668.42",6.27,"2,592.20",833.53,227.96,438.96,4557,46.41,21.07,"1,131.85"
Germany,107,85,79.44,"418,903.09",7.38,"3,914.98","1,316.66",303.23,546.59,84,58.33,29.76,"1,926.77"
France,93,66,70.97,"324,895.99",6.56,"3,493.51","1,045.35",110.13,578.60,66,57.58,24.24,"1,478.53"


## 5. Who keeps customers the longest?

In [6]:
from IPython.display import HTML, display
display(HTML(sections[4][1]))

## 6. When should we expect a second purchase?

In [7]:
from IPython.display import HTML, display
display(HTML(sections[5][1]))

Days after first order,Customers with a later order,All first-order customers,Returned by this point (%)
7,463,4934,9.38
14,712,4934,14.43
30,1118,4934,22.66
60,1856,4934,37.62
90,2301,4934,46.64


## 7. What is a customer worth after their first purchase?

In [8]:
from IPython.display import HTML, display
display(HTML(sections[6][1]))

Product code,Product,Customers,Average later net revenue (£),Median later net revenue (£)
85123A,WHITE HANGING HEART T-LIGHT HOLDER,836,"1,013.03",325.30
22138,BAKING SET 9 PIECE RETROSPOT,481,660.62,299.60
21212,PACK OF 72 RETRO SPOT CAKE CASES,477,"1,260.70",409.37
85099B,JUMBO BAG RED WHITE SPOTTY,401,"1,075.43",428.05
84879,ASSORTED COLOUR BIRD ORNAMENT,360,898.82,341.66
22139,RETRO SPOT TEA SET CERAMIC 11 PC,357,898.20,252.62
22423,REGENCY CAKESTAND 3 TIER,353,603.09,265.78
21034,REX CASH+CARRY JUMBO SHOPPER,333,515.23,172.24
21232,STRAWBERRY CERAMIC TRINKET BOX,326,"1,304.79",429.97
84991,60 TEATIME FAIRY CAKE CASES,319,910.41,409.37


## 8. Does the first order tell us who will return?

In [9]:
from IPython.display import HTML, display
display(HTML(sections[7][1]))

## 9. Do country and season change the picture?

In [10]:
from IPython.display import HTML, display
display(HTML(sections[8][1]))

Country,Customers,Returned within 90 days (%),Average first-order spending (£),Average later net revenue (£)
France,64,57.81,526.21,939.66
Germany,84,58.33,808.65,"1,118.11"
United Kingdom,4529,46.32,388.25,746.64
When the customer first bought,Customers,Returned within 90 days (%),Average first-order spending (£),Average later net revenue (£)
2009 Q4 (Oct–Dec),948,61.39,421.58,"2,008.99"
2010 Q1 (Jan–Mar),1181,49.96,399.99,715.51
2010 Q2 (Apr–Jun),814,39.07,414.69,491.53
2010 Q3 (Jul–Sep),583,47.51,444.01,554.30
2010 Q4 (Oct–Dec),774,35.66,435.02,325.67
2011 Q1 (Jan–Mar),374,38.77,396.06,490.68


## 10. How much can we predict after one order and after two?

In [11]:
from IPython.display import HTML, display
display(HTML(sections[9][1]))

Model,Validation Brier,Validation AUC
Logistic regression,0.2546,0.5320
Gradient boosting,0.2501,0.5611
Probabilistic baseline,0.2516,0.5000
Logistic regression + product identities,0.2515,0.5477
Gradient boosting + product identities,0.2526,0.5544
Model,Test Brier,Test AUC
"Gradient boosting (spend, items, variety)",0.2485,0.5853
Logistic regression + product identities,0.2492,0.5374
Information available,Algorithm,Validation Brier
First order only,Logistic regression,0.2555


## 11. What should the business do?

In [12]:
from IPython.display import HTML, display
display(HTML(sections[10][1]))